## Plot passenger dataset distribution

In [ ]:
import os
import glob
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# --- Configuration ---
LABEL_DIR = "/media/holidayj/Documents/data/for_v5/labels/train"
OUTPUT_DIR = "pics/passenger_relative"
# ---------------------

def analyze_relative_passenger(label_dir, output_dir):
    """
    Reads YOLO label files to extract relative dimensions
    and plots their distributions.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    data = []
    label_files = glob.glob(os.path.join(label_dir, "*.txt"))
    
    if not label_files:
        print(f"Error: No label files (.txt) found at {label_dir}")
        return

    print(f"Found {len(label_files)} label files. Processing...")

    for label_path in label_files:
        try:
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        # YOLO format: class x_center y_center rel_width rel_height
                        rel_w = float(parts[3])
                        rel_h = float(parts[4])
                        
                        # Calculate relative area
                        rel_area = rel_w * rel_h
                        
                        data.append({'width': rel_w, 'height': rel_h, 'area': rel_area})
        except Exception as e:
            print(f"Error reading label file {label_path}: {e}")
            continue

    if not data:
        print("No object data was successfully loaded.")
        return

    df = pd.DataFrame(data)
    
    print(f"\nSuccessfully loaded {len(df)} objects.")
    
    output_csv = os.path.join(output_dir, 'passenger_relative_dimensions.csv')
    df.to_csv(output_csv, index=False)
    print(f"Saved relative dimensions to '{output_csv}'")
    
    sns.set_theme(style="whitegrid")
    
    # --- Plotting as requested ---
    
    # Define metrics and scales
    metrics = ['width', 'height', 'area']
    scales = ['linear', 'log']
    
    # 1. Individual Plots
    print("Saving individual plots...")
    for metric in metrics:
        for scale in scales:
            plt.figure(figsize=(10, 6))
            sns.histplot(data=df, x=metric, bins=50)
            plt.yscale(scale)
            plt.title(f'Distribution of Relative {metric.capitalize()} ({scale.capitalize()} Scale)')
            plt.xlabel(f'Relative {metric.capitalize()} (0 to 1)')
            plt.ylabel(f'Count ({scale} scale)')
            plt.xlim(0, 1) # Set x-axis from 0 to 1
            plt.tight_layout()
            plot_file = os.path.join(output_dir, f'passenger_rel_{metric}_{scale}_scale.png')
            plt.savefig(plot_file)
            plt.clf()

    # 2. Combined Plots
    print("Saving combined plots...")
    
    # Melt the dataframe
    df_long = df.melt(
        value_vars=['width', 'height', 'area'], 
        var_name='Metric', 
        value_name='Relative Value'
    )
    
    for scale in scales:
        plt.figure(figsize=(12, 7))
        # Use element="step" and fill=False for a cleaner look
        sns.histplot(
            data=df_long, 
            x='Relative Value', 
            hue='Metric', 
            bins=50, 
            multiple="layer", 
            element="step", 
            fill=False
        )
        plt.yscale(scale)
        plt.title(f'Combined Relative Distributions ({scale.capitalize()} Scale)')
        plt.xlabel('Relative Value (0 to 1)')
        plt.ylabel(f'Count ({scale} scale)')
        plt.xlim(0, 1) # Set x-axis from 0 to 1
        plt.tight_layout()
        plot_file = os.path.join(output_dir, f'passenger_combined_rel_{scale}_scale.png')
        plt.savefig(plot_file)
        plt.clf()

    print(f"\nAll plots saved to {output_dir}")
    print("Done.")

# --- Main execution ---
if __name__ == "__main__":
    analyze_relative_passenger(LABEL_DIR, OUTPUT_DIR)

Found 5292 label files. Processing...

Successfully loaded 10764 objects.
Saved relative dimensions to 'pics/passenger_relative/passenger_relative_dimensions.csv'
Saving individual plots...
Saving combined plots...

All plots saved to pics/passenger_relative
Done.


<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1200x700 with 0 Axes>

<Figure size 1200x700 with 0 Axes>

## Plot coco dataset distribution

In [10]:
import os
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# --- Configuration ---
ANNOTATION_FILE = "/media/holidayj/Documents/data/datasets_for_object_detection/coco/annotations/instances_train2017.json"
OUTPUT_DIR      = "pics/coco_relative"
# ---------------------

def analyze_relative_coco(annotation_file, output_dir):
    """
    Reads a COCO JSON file, calculates relative dimensions,
    and plots their distributions.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"Loading annotation file: {annotation_file}...")
    try:
        with open(annotation_file, 'r') as f:
            coco_data = json.load(f)
    except Exception as e:
        print(f"Error loading JSON file: {e}")
        return

    # 1. Create a lookup map for image_id -> (width, height)
    print("Creating image dimension lookup...")
    image_dims = {}
    if 'images' not in coco_data:
        print("Error: 'images' key not found in JSON.")
        return
        
    for img in coco_data['images']:
        image_dims[img['id']] = (img['width'], img['height'])
    
    print(f"Found dimensions for {len(image_dims)} images.")

    # 2. Process annotations
    print("Processing annotations to calculate relative dimensions...")
    data = []
    if 'annotations' not in coco_data:
        print("Error: 'annotations' key not found in JSON.")
        return
        
    for ann in coco_data['annotations']:
        bbox = ann['bbox']
        image_id = ann['image_id']
        
        if image_id not in image_dims:
            print(f"Warning: No image metadata for image_id {image_id}. Skipping annotation {ann['id']}.")
            continue
            
        img_w, img_h = image_dims[image_id]
        
        # Ensure no division by zero if image has 0 width/height
        if img_w == 0 or img_h == 0:
            continue
            
        abs_w = bbox[2]
        abs_h = bbox[3]
        
        rel_w = abs_w / img_w
        rel_h = abs_h / img_h
        rel_area = rel_w * rel_h
        
        data.append({'width': rel_w, 'height': rel_h, 'area': rel_area})

    if not data:
        print("No annotation data was successfully loaded.")
        return

    df = pd.DataFrame(data)
    
    print(f"\nSuccessfully loaded and processed {len(df)} objects.")
    
    output_csv = os.path.join(output_dir, 'coco_relative_dimensions.csv')
    df.to_csv(output_csv, index=False)
    print(f"Saved relative dimensions to '{output_csv}'")
    
    sns.set_theme(style="whitegrid")
    
    # --- Plotting as requested ---
    
    # Define metrics and scales
    metrics = ['width', 'height', 'area']
    scales = ['linear', 'log']
    
    # 1. Individual Plots
    print("Saving individual plots...")
    for metric in metrics:
        for scale in scales:
            plt.figure(figsize=(10, 6))
            sns.histplot(data=df, x=metric, bins=50)
            plt.yscale(scale)
            plt.title(f'Distribution of Relative {metric.capitalize()} ({scale.capitalize()} Scale)')
            plt.xlabel(f'Relative {metric.capitalize()} (0 to 1)')
            plt.ylabel(f'Count ({scale} scale)')
            plt.xlim(0, 1) # Set x-axis from 0 to 1
            plt.tight_layout()
            plot_file = os.path.join(output_dir, f'coco_rel_{metric}_{scale}_scale.png')
            plt.savefig(plot_file)
            plt.clf()

    # 2. Combined Plots
    print("Saving combined plots...")
    
    # Melt the dataframe
    df_long = df.melt(
        value_vars=['width', 'height', 'area'], 
        var_name='Metric', 
        value_name='Relative Value'
    )
    
    for scale in scales:
        plt.figure(figsize=(12, 7))
        sns.histplot(
            data=df_long, 
            x='Relative Value', 
            hue='Metric', 
            bins=50, 
            multiple="layer", 
            element="step", 
            fill=False
        )
        plt.yscale(scale)
        plt.title(f'Combined Relative Distributions ({scale.capitalize()} Scale)')
        plt.xlabel('Relative Value (0 to 1)')
        plt.ylabel(f'Count ({scale} scale)')
        plt.xlim(0, 1) # Set x-axis from 0 to 1
        plt.tight_layout()
        plot_file = os.path.join(output_dir, f'coco_combined_rel_{scale}_scale.png')
        plt.savefig(plot_file)
        plt.clf()

    print(f"\nAll plots saved to {output_dir}")
    print("Done.")

# --- Main execution ---
if __name__ == "__main__":
    # Make sure you have the required libraries installed:
    # pip install pandas seaborn matplotlib
    
    analyze_relative_coco(ANNOTATION_FILE, OUTPUT_DIR)

Loading annotation file: /media/holidayj/Documents/data/datasets_for_object_detection/coco/annotations/instances_train2017.json...
Creating image dimension lookup...
Found dimensions for 118287 images.
Processing annotations to calculate relative dimensions...

Successfully loaded and processed 860001 objects.
Saved relative dimensions to 'pics/coco_relative/coco_relative_dimensions.csv'
Saving individual plots...
Saving combined plots...

All plots saved to pics/coco_relative
Done.


<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1200x700 with 0 Axes>

<Figure size 1200x700 with 0 Axes>

In [11]:
import pandas as pd
import numpy as np
from scipy.stats import entropy
import sys

# --- Configuration ---
PASSENGER_CSV = 'pics/passenger_relative/passenger_relative_dimensions.csv'
COCO_CSV = 'pics/coco_relative/coco_relative_dimensions.csv'

# We can adjust this, but 50 bins is a good start
NUM_BINS = 50 
# ---------------------

def calculate_entropy(data_series, bins):
    """
    Calculates the Shannon entropy for a continuous series
    by first discretizing it into bins.
    """
    # Get the counts for each bin
    counts, _ = np.histogram(data_series, bins=bins)
    
    # Get the total count
    total_count = counts.sum()
    if total_count == 0:
        return 0 # No data, so entropy is 0
        
    # Convert counts to probabilities
    probabilities = counts / total_count
    
    # Filter out probabilities that are 0 to avoid log(0)
    probabilities = probabilities[probabilities > 0]
    
    # Calculate entropy using base 2
    return entropy(probabilities, base=2)

def main():
    # Define our common bins (from 0 to 1)
    # We add 1 to NUM_BINS to get the correct number of bin *edges*
    bins = np.linspace(0, 1, num=NUM_BINS + 1)
    
    # --- Load Data ---
    try:
        df_pass = pd.read_csv(PASSENGER_CSV)
        df_coco = pd.read_csv(COCO_CSV)
    except FileNotFoundError as e:
        print(f"Error: File not found. {e}", file=sys.stderr)
        print("Please ensure you have run the analysis scripts and the CSV files are in:")
        print(f" - {PASSENGER_CSV}")
        print(f" - {COCO_CSV}")
        return

    print(f"Successfully loaded {len(df_pass)} passenger objects and {len(df_coco)} COCO objects.")
    print(f"Calculating entropy using {NUM_BINS} bins (from 0 to 1).")
    
    # --- Calculate Results ---
    results = {}
    metrics = ['width', 'height', 'area']
    
    for metric in metrics:
        entropy_pass = calculate_entropy(df_pass[metric], bins)
        entropy_coco = calculate_entropy(df_coco[metric], bins)
        
        results[metric] = {
            'Passenger': entropy_pass,
            'COCO': entropy_coco
        }

    # --- Print Results Table ---
    print("\n" + "="*53)
    print("      --- Entropy Comparison (in bits) ---")
    print("="*53)
    print(f"{'Metric':<10} | {'Passenger Entropy':<18} | {'COCO Entropy':<15}")
    print("-" * 53)
    
    for metric, values in results.items():
        print(f"{metric.capitalize():<10} | {values['Passenger']:<18.4f} | {values['COCO']:<15.4f}")
    
    print("-" * 53)
    
    print("\nNote: Higher entropy (bits) = More diversity and uncertainty in object sizes.")

if __name__ == "__main__":
    # Make sure you have the required libraries installed:
    # pip install pandas numpy scipy
    main()

Successfully loaded 10764 passenger objects and 860001 COCO objects.
Calculating entropy using 50 bins (from 0 to 1).

      --- Entropy Comparison (in bits) ---
Metric     | Passenger Entropy  | COCO Entropy   
-----------------------------------------------------
Width      | 3.6171             | 4.4688         
Height     | 4.4319             | 4.8477         
Area       | 3.1096             | 2.7502         
-----------------------------------------------------

Note: Higher entropy (bits) = More diversity and uncertainty in object sizes.
